# PathSnap - training pipeline (Kaggle "Save & Run All")

Runs every training/experiment step of the PathSnap repository end-to-end:

| Step | Script | What it produces |
|---|---|---|
| 1 | `01_eda.py` | dataset inventory, data-quality notes, EDA plots, Folium EDA maps |
| 2 | `02_build_features.py` | label alignment, candidate generation, feature tables |
| 3 | `03_train_models.py` | 7 models: 5-fold GroupKFold CV + student-test evaluation |
| 3c | `03c_compare_models.py` | comparison table + plots + confusion matrices |
| 4 | `04_bayes_opt.py` | Bayesian optimisation (Optuna TPE) of HistGradientBoosting, final model |
| 5 | `05_hmm_decoding.py` | HMM/Viterbi decoder tuning + classifier-only vs HMM comparison |
| 3b | `03b_learning_curves.py` | training/validation loss histories (HGB, MLP) |
| 6 | `06_make_maps.py` | Folium maps: GPS, network, reference, classifier-only, predicted route |

**Input:** add the Kaggle dataset that contains the `train/` and `student_test/` trip folders
(upload `Student folder.zip` as a dataset). Evaluator-testing data is **not** used.

**Output:** `/kaggle/working/pathsnap_outputs.zip` with `results/` and `models/` - copy them into the
repository (same folder names).

Only CPU is needed (no GPU/TPU). Internet is optional.

In [ ]:
# ---- parameters -------------------------------------------------------------
N_TRIALS = 25          # Bayesian-optimisation trials (each ~1-2 min on Kaggle CPU)
REPO = "/kaggle/working/PathSnap"

import os, sys, glob, json, shutil, subprocess, zipfile, platform
import sklearn, numpy, scipy, pandas, matplotlib
print("python", platform.python_version())
for m in (sklearn, numpy, scipy, pandas, matplotlib):
    print(m.__name__, m.__version__)
try:
    import optuna, folium, pyarrow
    print("optuna", optuna.__version__, "| folium", folium.__version__, "| pyarrow", pyarrow.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna", "folium", "pyarrow"], check=True)

# HistGradientBoosting(max_features=...) needs scikit-learn >= 1.4
major, minor = (int(x) for x in sklearn.__version__.split(".")[:2])
if (major, minor) < (1, 4):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "scikit-learn"], check=True)

In [ ]:
# ---- locate the data and lay out the repository -------------------------------
def find_data_root():
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "train" in dirs and "student_test" in dirs:
            return root
    return None

DATA_SRC = find_data_root()
if DATA_SRC is None:                       # dataset uploaded as a zip that Kaggle did not extract
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/kaggle/tmp/pathsnap_data")
    for root, dirs, _ in os.walk("/kaggle/tmp/pathsnap_data"):
        if "train" in dirs and "student_test" in dirs:
            DATA_SRC = root
            break
assert DATA_SRC, "No folder with 'train' and 'student_test' found - add the data as a Kaggle dataset."
print("data found at:", DATA_SRC)

for d in ["pathsnap", "scripts", "data/evaluator_test", "results", "models", "logs"]:
    os.makedirs(os.path.join(REPO, d), exist_ok=True)
for part in ["train", "student_test"]:
    dst = os.path.join(REPO, "data", part)
    if not os.path.exists(dst):
        os.symlink(os.path.join(DATA_SRC, part), dst)
    print(part, len(os.listdir(dst)), "trips")

## Source code
Each cell below writes one module/script of the repository (identical to the GitHub version).

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/__init__.py
"""PathSnap: map-matching of noisy GPS trajectories onto a road network.

Pipeline: raw trip files -> local metric projection -> candidate road segments
-> candidate features -> classifier probabilities -> HMM/Viterbi decoding.
"""

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/config.py
"""Central configuration. All paths are relative to the repository root."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
TRAIN_DIR = DATA_DIR / "train"
STUDENT_TEST_DIR = DATA_DIR / "student_test"
EVALUATOR_TEST_DIR = DATA_DIR / "evaluator_test"   # filled on evaluation day, never used in development
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
FEATURES_DIR = ROOT / "results" / "features"

RANDOM_SEED = 42

# Candidate generation (chosen from the training-partition EDA, see results/eda/)
CANDIDATE_RADIUS_M = 50.0     # search radius around every GPS point
MAX_CANDIDATES = 8            # keep the K nearest road segments inside the radius
DENSIFY_STEP_M = 10.0         # sample spacing used to index segments in the KD-tree

# Graph cropping: keep only road nodes within this margin of the trajectory bbox
GRAPH_MARGIN_M = 300.0

# Cross-validation
N_FOLDS = 5

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/geo.py
"""Geometry helpers.

Longitude/latitude are angles, so Euclidean distance in degrees is not a
physical distance. Every trip is projected onto a local tangent plane
(equirectangular projection centred on the trip's mean position):

    x = R * (lon - lon0) * cos(lat0)      y = R * (lat - lat0)     [radians]

For trips spanning a few tens of km this keeps local distances (tens of
metres, the scale map-matching works at) accurate to well below 1 %, needs no
external projection library and works anywhere on the globe (the dataset
contains trips on different continents, so one fixed UTM zone would not fit).
"""
import numpy as np

EARTH_R = 6_371_008.8  # mean Earth radius in metres


class LocalProjection:
    def __init__(self, lon0, lat0):
        self.lon0 = lon0
        self.lat0 = lat0
        self.kx = EARTH_R * np.cos(np.radians(lat0)) * np.pi / 180.0
        self.ky = EARTH_R * np.pi / 180.0

    def forward(self, lon, lat):
        lon = np.asarray(lon, float)
        lat = np.asarray(lat, float)
        return np.column_stack([(lon - self.lon0) * self.kx, (lat - self.lat0) * self.ky])

    def inverse(self, xy):
        xy = np.asarray(xy, float)
        return np.column_stack([xy[:, 0] / self.kx + self.lon0, xy[:, 1] / self.ky + self.lat0])


def haversine_m(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))


def point_segment(p, a, b):
    """Vectorised point-to-segment projection.

    p, a, b: (..., 2) arrays (broadcastable).
    Returns (distance, t_clipped, t_raw) where t is the position along a->b.
    """
    ab = b - a
    ap = p - a
    L2 = np.sum(ab * ab, axis=-1)
    L2 = np.where(L2 < 1e-12, 1e-12, L2)
    t_raw = np.sum(ap * ab, axis=-1) / L2
    t = np.clip(t_raw, 0.0, 1.0)
    proj = a + t[..., None] * ab
    d = np.linalg.norm(p - proj, axis=-1)
    return d, t, t_raw


def bearing_xy(dx, dy):
    """Compass-style bearing in degrees [0,360) of a vector in the local plane."""
    return (np.degrees(np.arctan2(dx, dy)) + 360.0) % 360.0


def angle_diff(a, b):
    """Absolute smallest difference between two bearings, in [0,180]."""
    d = np.abs(a - b) % 360.0
    return np.minimum(d, 360.0 - d)

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/data_io.py
"""Reading raw PathSnap trip folders.

Each trip folder <id>/ contains:
  <id>.track : lon lat t      (whitespace separated, no header)
  <id>.nodes : lon lat        (row index = node_id)
  <id>.arcs  : from to        (row index = arc_id, directed)
  <id>.route : arc_id         (chronological reference route; optional)
"""
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np

from . import geo


@dataclass
class Trip:
    trip_id: str
    track: np.ndarray            # (N,3) lon, lat, t
    nodes: np.ndarray            # (V,2) lon, lat
    arcs: np.ndarray             # (E,2) from_node, to_node
    route: Optional[np.ndarray]  # (M,) arc ids or None
    proj: "geo.LocalProjection" = field(default=None)
    # projected (metres) versions, filled by .project()
    xy: np.ndarray = field(default=None)        # (N,2)
    node_xy: np.ndarray = field(default=None)   # (V,2)

    def project(self):
        lon0 = float(np.mean(self.track[:, 0]))
        lat0 = float(np.mean(self.track[:, 1]))
        self.proj = geo.LocalProjection(lon0, lat0)
        self.xy = self.proj.forward(self.track[:, 0], self.track[:, 1])
        self.node_xy = self.proj.forward(self.nodes[:, 0], self.nodes[:, 1])
        return self

    @property
    def times(self):
        return self.track[:, 2]


def list_trips(partition_dir):
    partition_dir = Path(partition_dir)
    if not partition_dir.exists():
        return []
    return sorted(p.name for p in partition_dir.iterdir()
                  if p.is_dir() and (p / f"{p.name}.track").exists())


def _load(path, dtype=float, ncols=None):
    arr = np.loadtxt(path, dtype=dtype, ndmin=2 if ncols else 1)
    return arr


def load_trip(partition_dir, trip_id, with_route=True, project=True) -> Trip:
    d = Path(partition_dir) / trip_id
    track = np.loadtxt(d / f"{trip_id}.track", ndmin=2)
    nodes = np.loadtxt(d / f"{trip_id}.nodes", ndmin=2)
    arcs = np.loadtxt(d / f"{trip_id}.arcs", dtype=np.int64, ndmin=2)
    route = None
    rp = d / f"{trip_id}.route"
    if with_route and rp.exists():
        route = np.loadtxt(rp, dtype=np.int64, ndmin=1)
    trip = Trip(trip_id, track, nodes, arcs, route)
    trip = clean_track(trip)
    if project:
        trip.project()
    return trip


def clean_track(trip: Trip) -> Trip:
    """Minimal, label-free cleaning applied identically to every partition:
    drop rows with non-finite values / implausible coordinates, sort by time,
    and drop exact duplicate timestamps (keep first)."""
    tr = trip.track
    ok = np.all(np.isfinite(tr), axis=1)
    ok &= (np.abs(tr[:, 0]) <= 180) & (np.abs(tr[:, 1]) <= 90)
    tr = tr[ok]
    order = np.argsort(tr[:, 2], kind="stable")
    tr = tr[order]
    keep = np.ones(len(tr), bool)
    keep[1:] = np.diff(tr[:, 2]) > 0
    trip.track = tr[keep]
    return trip

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/network.py
"""Road network utilities: segment geometry, spatial index and shortest paths."""
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.spatial import cKDTree

from . import config, geo


class RoadNetwork:
    """Directed road graph of one trip, in the trip's local metric frame.

    Only arcs whose end nodes lie within GRAPH_MARGIN_M of the trajectory's
    bounding box are indexed (the node files cover a much larger area than the
    trajectory). Arc ids are always the ORIGINAL row indices of the .arcs file.
    """

    def __init__(self, trip, margin_m=config.GRAPH_MARGIN_M, step_m=config.DENSIFY_STEP_M):
        self.trip = trip
        nxy = trip.node_xy
        arcs = trip.arcs
        lo = trip.xy.min(0) - margin_m
        hi = trip.xy.max(0) + margin_m
        inside = np.all((nxy >= lo) & (nxy <= hi), axis=1)
        valid = (arcs[:, 0] >= 0) & (arcs[:, 1] >= 0) & (arcs[:, 0] < len(nxy)) & (arcs[:, 1] < len(nxy))
        keep = valid.copy()
        keep[valid] = inside[arcs[valid, 0]] & inside[arcs[valid, 1]]
        self.arc_ids = np.nonzero(keep)[0]                 # indexed arcs (original ids)
        self.n_nodes = len(nxy)
        self.A = nxy[arcs[:, 0].clip(0, len(nxy) - 1)]      # start point of every arc
        self.B = nxy[arcs[:, 1].clip(0, len(nxy) - 1)]      # end point of every arc
        self.length = np.linalg.norm(self.B - self.A, axis=1)
        d = self.B - self.A
        self.bearing = geo.bearing_xy(d[:, 0], d[:, 1])
        self.out_deg = np.bincount(arcs[keep, 0], minlength=self.n_nodes)
        self.in_deg = np.bincount(arcs[keep, 1], minlength=self.n_nodes)
        # does the opposite-direction arc exist? (two-way road indicator)
        key = arcs[keep, 0].astype(np.int64) * self.n_nodes + arcs[keep, 1]
        rkey = arcs[keep, 1].astype(np.int64) * self.n_nodes + arcs[keep, 0]
        self.twoway = np.zeros(len(arcs), bool)
        self.twoway[self.arc_ids] = np.isin(rkey, key)
        self._build_index(step_m)
        self._build_graph(keep)

    # ---------------- spatial index ----------------
    def _build_index(self, step_m):
        ids = self.arc_ids
        n_samp = np.maximum(2, np.ceil(self.length[ids] / step_m).astype(int) + 1)
        rep = np.repeat(np.arange(len(ids)), n_samp)
        offs = np.concatenate([np.linspace(0, 1, k) for k in n_samp]) if len(ids) else np.zeros(0)
        pts = self.A[ids][rep] + offs[:, None] * (self.B[ids][rep] - self.A[ids][rep])
        self.sample_arc = ids[rep]
        self.tree = cKDTree(pts) if len(pts) else None
        self.half_step = step_m / 2.0

    def candidates(self, xy, radius=config.CANDIDATE_RADIUS_M, k=config.MAX_CANDIDATES):
        """For each point return the k nearest arcs within `radius` (exact
        point-to-segment distance). Returns list of (arc_ids, dist, t, t_raw)."""
        out = []
        if self.tree is None:
            return [(np.zeros(0, int), np.zeros(0), np.zeros(0), np.zeros(0)) for _ in xy]
        hits = self.tree.query_ball_point(xy, r=radius + self.half_step + 1.0)
        for p, h in zip(xy, hits):
            if not h:
                out.append((np.zeros(0, int), np.zeros(0), np.zeros(0), np.zeros(0)))
                continue
            a = np.unique(self.sample_arc[h])
            d, t, traw = geo.point_segment(p[None, :], self.A[a], self.B[a])
            m = d <= radius
            a, d, t, traw = a[m], d[m], t[m], traw[m]
            o = np.argsort(d, kind="stable")[:k]
            out.append((a[o], d[o], t[o], traw[o]))
        return out

    # ---------------- graph / shortest paths ----------------
    def _build_graph(self, keep):
        arcs = self.trip.arcs
        u = arcs[keep, 0]
        v = arcs[keep, 1]
        w = np.maximum(self.length[keep], 0.01)
        # csr_matrix would SUM duplicate (u,v) entries; keep the shortest parallel arc instead
        key = u.astype(np.int64) * self.n_nodes + v
        order = np.lexsort((w, key))
        first = np.ones(len(order), bool)
        first[1:] = key[order][1:] != key[order][:-1]
        sel = order[first]
        u, v, w = u[sel], v[sel], w[sel]
        self.uv_to_arc = dict(zip(zip(u.tolist(), v.tolist()), self.arc_ids[sel].tolist()))
        # compact node indexing (only nodes used by indexed arcs) -> much smaller Dijkstra outputs
        used = np.unique(np.concatenate([u, v]))
        self.local = np.full(self.n_nodes, -1, dtype=np.int64)
        self.local[used] = np.arange(len(used))
        self.global_node = used
        m = len(used)
        self.G = csr_matrix((w, (self.local[u], self.local[v])), shape=(m, m))
        self._sp_cache = {}

    def distances_from(self, node, limit):
        """Shortest network distances from ORIGINAL node id `node`, up to `limit` metres.
        Returns dict {original_node_id: distance}. Results are cached per (node, limit)."""
        key = (int(node), float(limit))
        hit = self._sp_cache.get(key)
        if hit is not None:
            return hit
        ln = self.local[node] if 0 <= node < self.n_nodes else -1
        if ln < 0:
            res = {int(node): 0.0}
        else:
            d = dijkstra(self.G, directed=True, indices=int(ln), limit=limit)
            idx = np.nonzero(np.isfinite(d))[0]
            res = dict(zip(self.global_node[idx].tolist(), d[idx].tolist()))
        if len(self._sp_cache) > 20000:
            self._sp_cache.clear()
        self._sp_cache[key] = res
        return res

    def shortest_path_nodes(self, src, dst, limit=5000.0):
        """Node sequence (original ids) of the shortest path src->dst, or None."""
        ls, ld = self.local[src], self.local[dst]
        if ls < 0 or ld < 0:
            return None
        if ls == ld:
            return [int(src)]
        d, pred = dijkstra(self.G, directed=True, indices=int(ls), limit=limit, return_predecessors=True)
        if not np.isfinite(d[ld]):
            return None
        path = [int(ld)]
        while path[-1] != ls:
            path.append(int(pred[path[-1]]))
        return [int(self.global_node[p]) for p in reversed(path)]

    def arc_connected(self, a, b):
        """True if arc b starts where arc a ends (or a == b)."""
        arcs = self.trip.arcs
        return (a == b) or (arcs[a, 1] == arcs[b, 0])

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/labels.py
"""Deriving observation-level reference labels from the .route file.

The .route file is a chronological list of arc ids, NOT one label per GPS row
(a trip with 1 315 GPS rows has a 525-arc route). To obtain a reference road
segment for every observation we solve a monotone alignment problem:

    choose j_0 <= j_1 <= ... <= j_{N-1}   (indices into the route)
    minimising  sum_i dist(p_i, segment(route[j_i]))

i.e. every GPS point is assigned to a route arc, and the assignment may only
move forward along the route (the vehicle never travels backwards in time).
Solved exactly by dynamic programming with a running prefix-minimum,
O(N*M) time.

These labels are used ONLY for training and for server-side scoring; they
never enter the prediction pipeline.
"""
import numpy as np

from . import geo


def align_route(trip, chunk=400):
    route = trip.route
    if route is None or len(route) == 0:
        return None, None
    nxy = trip.node_xy
    A = nxy[trip.arcs[route, 0]]
    B = nxy[trip.arcs[route, 1]]
    N, M = len(trip.xy), len(route)
    back = np.empty((N, M), dtype=np.int32)
    prev = None
    for s in range(0, N, chunk):
        P = trip.xy[s:s + chunk]
        C, _, _ = geo.point_segment(P[:, None, :], A[None, :, :], B[None, :, :])
        C = C.astype(np.float64)
        for r in range(len(P)):
            i = s + r
            if prev is None:
                cur = C[r].copy()
                back[i] = np.arange(M)
            else:
                # prefix minimum of previous row and its argmin
                idx = np.arange(M)
                is_new_min = np.empty(M, bool)
                pm = np.minimum.accumulate(prev)
                is_new_min[0] = True
                is_new_min[1:] = prev[1:] <= pm[:-1]
                arg = np.maximum.accumulate(np.where(is_new_min, idx, 0))
                back[i] = arg
                cur = C[r] + pm
            prev = cur
    j = int(np.argmin(prev))
    path = np.empty(N, dtype=np.int64)
    for i in range(N - 1, -1, -1):
        path[i] = j
        j = back[i, j]
    labels = route[path]
    dist, _, _ = geo.point_segment(trip.xy, nxy[trip.arcs[labels, 0]], nxy[trip.arcs[labels, 1]])
    return labels, dist

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/features.py
"""Candidate generation and feature engineering.

Problem framing
---------------
Every GPS observation i gets a candidate set C_i = the K nearest road
segments (arcs) within R metres. Each (observation, candidate) pair becomes
one row; the binary target is 1 if the candidate is the observation's
reference arc. A classifier scores every pair; per observation the scores
are normalised over C_i to give P(arc | observation), which (a) gives the
classifier-only prediction (arg-max) and (b) is the HMM emission.

Features (all computed from GPS + road network only, never from labels)
-----------------------------------------------------------------------
dist            perpendicular distance point -> segment (m)
dist_rank       rank of the candidate by distance inside C_i (0 = nearest)
dist_gap        dist - min(dist in C_i)
dist_ratio      dist / (min dist + 1)
n_cands         |C_i|  (dense road areas are harder)
head_diff       |GPS heading - arc bearing| in [0,180] (directed arcs!)
head_diff_w     same, using a +/-3 obs heading (more robust at low speed)
head_valid      1 if the vehicle moved enough for a reliable heading
speed           GPS speed (m/s, central difference)
arc_len         segment length (m)
t_pos           projected position along the arc in [0,1]
t_outside       metres the projection falls beyond the segment ends
dist_prev/next  distance of the previous / next GPS point to the same arc
dist_win        mean distance of points i-3..i+3 to the arc (temporal context)
out_deg/in_deg  degree of the arc's end/start node (junction complexity)
twoway          1 if the opposite-direction arc exists
dt_prev/dt_next time gap to previous/next observation (s)
"""
import numpy as np
import pandas as pd

from . import config, geo

FEATURES = [
    "dist", "dist_rank", "dist_gap", "dist_ratio", "n_cands",
    "head_diff", "head_diff_w", "head_valid", "speed",
    "arc_len", "t_pos", "t_outside",
    "dist_prev", "dist_next", "dist_win",
    "out_deg", "in_deg", "twoway", "dt_prev", "dt_next",
]


def _headings(xy, t, w):
    n = len(xy)
    i0 = np.clip(np.arange(n) - w, 0, n - 1)
    i1 = np.clip(np.arange(n) + w, 0, n - 1)
    d = xy[i1] - xy[i0]
    dist = np.linalg.norm(d, axis=1)
    dt = np.maximum(t[i1] - t[i0], 1e-6)
    return geo.bearing_xy(d[:, 0], d[:, 1]), dist, dist / dt


def build_candidate_table(trip, net, labels=None,
                          radius=config.CANDIDATE_RADIUS_M, k=config.MAX_CANDIDATES):
    """Return a DataFrame with one row per (observation, candidate arc)."""
    xy, t = trip.xy, trip.times
    n = len(xy)
    cands = net.candidates(xy, radius, k)
    head1, mv1, speed = _headings(xy, t, 1)
    head3, mv3, _ = _headings(xy, t, 3)
    dt = np.diff(t)
    dt_prev = np.concatenate([[0.0], dt])
    dt_next = np.concatenate([dt, [0.0]])

    obs = np.concatenate([np.full(len(c[0]), i) for i, c in enumerate(cands)]).astype(np.int64)
    if len(obs) == 0:
        return pd.DataFrame(columns=["obs", "arc"] + FEATURES)
    arc = np.concatenate([c[0] for c in cands]).astype(np.int64)
    dist = np.concatenate([c[1] for c in cands])
    tpos = np.concatenate([c[2] for c in cands])
    traw = np.concatenate([c[3] for c in cands])
    rank = np.concatenate([np.arange(len(c[0])) for c in cands])
    ncand = np.concatenate([np.full(len(c[0]), len(c[0])) for c in cands])
    dmin = np.concatenate([np.full(len(c[0]), c[1][0]) for c in cands if len(c[0])])

    A, B = net.A[arc], net.B[arc]
    L = np.maximum(net.length[arc], 1e-6)
    t_out = np.where(traw < 0, -traw, np.where(traw > 1, traw - 1, 0.0)) * L

    def dist_at(offset):
        j = np.clip(obs + offset, 0, n - 1)
        d, _, _ = geo.point_segment(xy[j], A, B)
        return d

    win = [dist_at(o) for o in range(-3, 4)]
    df = pd.DataFrame({
        "obs": obs, "arc": arc,
        "dist": dist, "dist_rank": rank, "dist_gap": dist - dmin,
        "dist_ratio": dist / (dmin + 1.0), "n_cands": ncand,
        "head_diff": geo.angle_diff(head1[obs], net.bearing[arc]),
        "head_diff_w": geo.angle_diff(head3[obs], net.bearing[arc]),
        "head_valid": (mv1[obs] > 2.0).astype(np.int8),
        "speed": speed[obs],
        "arc_len": net.length[arc], "t_pos": tpos, "t_outside": t_out,
        "dist_prev": win[2], "dist_next": win[4], "dist_win": np.mean(win, axis=0),
        "out_deg": net.out_deg[net.trip.arcs[arc, 1]], "in_deg": net.in_deg[net.trip.arcs[arc, 0]],
        "twoway": net.twoway[arc].astype(np.int8),
        "dt_prev": dt_prev[obs], "dt_next": dt_next[obs],
    })
    # heading is meaningless when stationary: replace with a neutral 90 deg
    stat = df["head_valid"].values == 0
    df.loc[stat, "head_diff"] = 90.0
    df.loc[mv3[obs] <= 2.0, "head_diff_w"] = 90.0
    if labels is not None:
        df["y"] = (arc == labels[obs]).astype(np.int8)
    df.insert(0, "trip", trip.trip_id)
    return df

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/visual.py
"""Folium map generation (used by the EDA/report scripts and by the API)."""
import folium
import numpy as np
from scipy.spatial import cKDTree

COLORS = {"gps": "#1f77b4", "network": "#9e9e9e", "reference": "#2ca02c",
          "predicted": "#d62728", "classifier": "#ff7f0e"}


def _arcs_to_lines(trip, arc_ids):
    """Merge a chronological arc sequence into polylines (break where not connected)."""
    lines, cur = [], []
    prev_to = None
    for a in arc_ids:
        u, v = trip.arcs[a]
        if prev_to is not None and u != prev_to and (not cur or a != cur[-1]):
            if len(cur) > 1:
                lines.append(cur)
            cur = []
        if not cur:
            cur.append(u)
        cur.append(v)
        prev_to = v
    if len(cur) > 1:
        lines.append(cur)
    return [[(trip.nodes[n, 1], trip.nodes[n, 0]) for n in line] for line in lines]


def dedupe_consecutive(seq):
    seq = list(seq)
    return [a for i, a in enumerate(seq) if i == 0 or a != seq[i - 1]]


def _legend(items):
    rows = "".join(
        f'<div><span style="display:inline-block;width:18px;height:4px;background:{c};'
        f'margin-right:6px;vertical-align:middle"></span>{name}</div>' for name, c in items)
    return folium.Element(
        '<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;'
        'padding:8px 10px;border:1px solid #888;border-radius:6px;font:12px sans-serif">'
        f'<b>Legend</b>{rows}</div>')


def trip_map(trip, show_network=True, reference_arcs=None, predicted_arcs=None,
             classifier_arcs=None, title=None, network_radius_m=120, max_gps_points=4000):
    lat, lon = trip.track[:, 1], trip.track[:, 0]
    m = folium.Map(location=[lat.mean(), lon.mean()], zoom_start=14, tiles="OpenStreetMap",
                   control_scale=True)
    if title:
        m.get_root().html.add_child(folium.Element(
            f'<div style="position:fixed;top:10px;left:60px;z-index:9999;background:white;'
            f'padding:4px 8px;border:1px solid #888;border-radius:4px;font:13px sans-serif"><b>{title}</b></div>'))
    legend = []
    if show_network:
        fg = folium.FeatureGroup(name="Road network (near trajectory)", show=True)
        tree = cKDTree(trip.xy)
        mid = (trip.node_xy[trip.arcs[:, 0]] + trip.node_xy[trip.arcs[:, 1]]) / 2
        near = np.nonzero(tree.query(mid, distance_upper_bound=network_radius_m)[0] < np.inf)[0]
        for a in near:
            u, v = trip.arcs[a]
            folium.PolyLine([(trip.nodes[u, 1], trip.nodes[u, 0]), (trip.nodes[v, 1], trip.nodes[v, 0])],
                            color=COLORS["network"], weight=2, opacity=0.6).add_to(fg)
        fg.add_to(m)
        legend.append(("Road network", COLORS["network"]))
    if reference_arcs is not None:
        fg = folium.FeatureGroup(name="Reference route", show=True)
        for line in _arcs_to_lines(trip, dedupe_consecutive(reference_arcs)):
            folium.PolyLine(line, color=COLORS["reference"], weight=7, opacity=0.6).add_to(fg)
        fg.add_to(m)
        legend.append(("Reference route", COLORS["reference"]))
    if classifier_arcs is not None:
        fg = folium.FeatureGroup(name="Classifier-only prediction", show=False)
        for line in _arcs_to_lines(trip, dedupe_consecutive(classifier_arcs)):
            folium.PolyLine(line, color=COLORS["classifier"], weight=3, opacity=0.9, dash_array="6").add_to(fg)
        fg.add_to(m)
        legend.append(("Classifier-only prediction", COLORS["classifier"]))
    if predicted_arcs is not None:
        fg = folium.FeatureGroup(name="Predicted route (HMM/Viterbi)", show=True)
        for line in _arcs_to_lines(trip, dedupe_consecutive(predicted_arcs)):
            folium.PolyLine(line, color=COLORS["predicted"], weight=3.5, opacity=0.9).add_to(fg)
        fg.add_to(m)
        legend.append(("Predicted route", COLORS["predicted"]))
    fg = folium.FeatureGroup(name="Raw GPS observations", show=True)
    step = max(1, int(np.ceil(len(lat) / max_gps_points)))
    for i in range(0, len(lat), step):
        folium.CircleMarker((lat[i], lon[i]), radius=2, color=COLORS["gps"], fill=True,
                            fill_opacity=0.8, weight=1, tooltip=f"obs {i}, t={trip.track[i, 2]:.0f}s").add_to(fg)
    fg.add_to(m)
    legend.append(("GPS observation", COLORS["gps"]))
    folium.Marker((lat[0], lon[0]), tooltip="Start", icon=folium.Icon(color="green", icon="play")).add_to(m)
    folium.Marker((lat[-1], lon[-1]), tooltip="End", icon=folium.Icon(color="red", icon="stop")).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(_legend(legend))
    m.fit_bounds([[lat.min(), lon.min()], [lat.max(), lon.max()]])
    return m

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/metrics.py
"""Evaluation metrics.

Definitions (used identically for every model and for the API):

* Observation-level Top-1 accuracy: for every GPS observation take the
  candidate with the highest score; correct if it equals the reference arc.
  Denominator = ALL observations (observations whose reference arc is not in
  the candidate set, or that have no candidates, count as errors).
* Candidate-level precision / recall / F1 / Macro-F1: every (observation,
  candidate) row is a binary decision; the predicted positive is the top-1
  candidate of its observation. Macro-F1 = mean of the F1 of class 1
  ("is the true segment") and class 0 ("is not"). A per-arc multi-class
  Macro-F1 is not meaningful because arc ids are trip-local.
* Log loss: binary cross-entropy of the classifier's candidate probabilities.
* Candidate miss rate: fraction of observations whose reference arc is not in
  the candidate set (an upper bound on achievable accuracy).
* Route continuity: fraction of consecutive observation pairs whose predicted
  arcs are identical or directly connected (end node of one = start node of
  the next). 1.0 = a physically drivable sequence.
* Route (sequence-level) precision / recall / F1: overlap, weighted by arc
  length, between the set of arcs in the predicted route and the reference
  route.
"""
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, log_loss, precision_score, recall_score


def top1_from_scores(obs, scores):
    """Return a boolean mask selecting the arg-max row per observation."""
    df = pd.DataFrame({"obs": np.asarray(obs), "s": np.asarray(scores)})
    idx = df.groupby("obs", sort=False)["s"].idxmax().values
    mask = np.zeros(len(df), bool)
    mask[idx] = True
    return mask


def candidate_metrics(y, top1_mask, proba=None):
    y = np.asarray(y).astype(int)
    pred = top1_mask.astype(int)
    out = {
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
    }
    if proba is not None:
        out["log_loss"] = log_loss(y, np.clip(proba, 1e-15, 1 - 1e-15), labels=[0, 1])
    return out


def route_continuity(arcs_seq, arcs_table):
    s = np.asarray(arcs_seq)
    valid = s >= 0
    a, b = s[:-1], s[1:]
    ok = valid[:-1] & valid[1:]
    if ok.sum() == 0:
        return float("nan")
    a, b = a[ok], b[ok]
    cont = (a == b) | (arcs_table[a, 1] == arcs_table[b, 0])
    return float(cont.mean())


def route_overlap(pred_arcs, ref_arcs, arc_length):
    p = set(int(a) for a in pred_arcs if a >= 0)
    r = set(int(a) for a in ref_arcs)
    if not p or not r:
        return {"route_precision": 0.0, "route_recall": 0.0, "route_f1": 0.0}
    inter = sum(arc_length[a] for a in p & r)
    prec = inter / max(sum(arc_length[a] for a in p), 1e-9)
    rec = inter / max(sum(arc_length[a] for a in r), 1e-9)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return {"route_precision": prec, "route_recall": rec, "route_f1": f1}


def observation_accuracy(pred_arcs, ref_arcs):
    pred_arcs = np.asarray(pred_arcs)
    ref_arcs = np.asarray(ref_arcs)
    return float(np.mean(pred_arcs == ref_arcs))

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/models.py
"""Model zoo: every model experimented with, with its initial configuration.

All models are binary classifiers on (observation, candidate) rows and expose
predict_proba, so their outputs can be used directly as HMM emissions.
"""
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from .config import RANDOM_SEED

# family, short description, factory, max training rows (None = all; subsampling
# is only used where the algorithm's cost would otherwise be impractical)
MODEL_SPECS = {
    "LogisticRegression": dict(
        family="Linear / probabilistic",
        config="StandardScaler + L2 logistic regression, C=1.0, lbfgs, max_iter=1000",
        factory=lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)),
        max_rows=None),
    "GaussianNB": dict(
        family="Linear / probabilistic",
        config="Gaussian Naive Bayes, var_smoothing=1e-9",
        factory=lambda: GaussianNB(),
        max_rows=None),
    "KNN": dict(
        family="Instance-based",
        config="StandardScaler + KNN k=25, distance-weighted, Euclidean; fit on 100k-row sample",
        factory=lambda: make_pipeline(StandardScaler(),
                                      KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)),
        max_rows=100_000),
    "DecisionTree": dict(
        family="Tree-based",
        config="CART, gini, max_depth=12, min_samples_leaf=50",
        factory=lambda: DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, random_state=RANDOM_SEED),
        max_rows=None),
    "RandomForest": dict(
        family="Ensemble (bagging)",
        config="100 trees, max_depth=18, min_samples_leaf=5, max_features=sqrt, max_samples=0.25",
        factory=lambda: RandomForestClassifier(n_estimators=100, max_depth=18, min_samples_leaf=5,
                                               max_samples=0.25, n_jobs=-1, random_state=RANDOM_SEED),
        max_rows=None),
    "HistGradientBoosting": dict(
        family="Boosting",
        config="300 iterations, learning_rate=0.1, max_leaf_nodes=31, l2=0, log-loss",
        factory=lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31,
                                                       early_stopping=False, random_state=RANDOM_SEED),
        max_rows=None),
    "MLP": dict(
        family="Neural network",
        config="StandardScaler + MLP (64,32) ReLU, Adam lr=1e-3, alpha=1e-4, batch=1024, 20 epochs; fit on 300k-row sample",
        factory=lambda: make_pipeline(StandardScaler(),
                                      MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, batch_size=1024,
                                                    learning_rate_init=1e-3, max_iter=20, random_state=RANDOM_SEED)),
        max_rows=300_000),
}

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/training.py
"""Shared helpers for loading the feature tables and evaluating a fitted model."""
import time

import numpy as np
import pandas as pd

from . import config, data_io, metrics
from .features import FEATURES


def label_ok_trips(max_median_m=30.0):
    """Training trips whose reference route actually follows the GPS (EDA finding)."""
    st = pd.read_csv(config.FEATURES_DIR / "candidate_stats.csv", dtype={"trip": str})
    st = st[st.partition == "train"]
    return set(st.loc[st.label_dist_median_m <= max_median_m, "trip"])


def load_training_rows():
    """Training rows: label-reliable trips, observations whose reference arc is a candidate."""
    df = pd.read_parquet(config.FEATURES_DIR / "train_candidates.parquet")
    df = df[df.trip.isin(label_ok_trips())]
    pos = df.loc[df.y == 1, ["trip", "obs"]].drop_duplicates()
    df = df.merge(pos, on=["trip", "obs"], how="inner")
    return df.reset_index(drop=True)


def load_student_test():
    df = pd.read_parquet(config.FEATURES_DIR / "student_test_candidates.parquet")
    obs = pd.read_csv(config.FEATURES_DIR / "student_test_observations.csv", dtype={"trip": str})
    return df, obs


def subsample(X, y, max_rows, seed=config.RANDOM_SEED):
    if max_rows is None or len(X) <= max_rows:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), max_rows, replace=False)
    return X[idx], y[idx]


def evaluate_rows(df, proba):
    """Row-level metrics on a candidate table whose rows all have a true candidate (CV)."""
    key = df.trip.astype(str) + "_" + df.obs.astype(str)
    mask = metrics.top1_from_scores(key.values, proba)
    m = metrics.candidate_metrics(df.y.values, mask, proba)
    m["top1_acc"] = float(df.y.values[mask].sum() / key.nunique())
    return m


def evaluate_partition(df, obs_table, proba, arcs_by_trip):
    """Full evaluation on a partition: every GPS observation counts.

    Returns (summary dict, per-observation predicted arcs dict trip->array)."""
    key = df.trip.astype(str) + "_" + df.obs.astype(str)
    mask = metrics.top1_from_scores(key.values, proba)
    cm = metrics.candidate_metrics(df.y.values, mask, proba)
    top = df.loc[mask, ["trip", "obs", "arc"]]
    preds, conts, correct, total = {}, [], 0, 0
    for tid, g in obs_table.groupby("trip"):
        p = np.full(len(g), -1, dtype=np.int64)
        t = top[top.trip == tid]
        p[t.obs.values] = t.arc.values
        preds[tid] = p
        correct += int(np.sum(p == g.ref_arc.values))
        total += len(g)
        conts.append(metrics.route_continuity(p, arcs_by_trip[tid]))
    cm["top1_acc"] = correct / total
    cm["route_continuity"] = float(np.nanmean(conts))
    return cm, preds


def arcs_tables(partition_dir, trip_ids):
    return {t: np.loadtxt(partition_dir / t / f"{t}.arcs", dtype=np.int64, ndmin=2) for t in trip_ids}


def timed_predict(model, X):
    t0 = time.perf_counter()
    p = model.predict_proba(X)[:, 1]
    return p, time.perf_counter() - t0


__all__ = ["FEATURES"]

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/hmm.py
"""HMM + Viterbi route reconstruction.

Hidden states
    At observation i the hidden state is "the vehicle is on candidate arc c,
    at the projected position t_c of the GPS point", for c in the candidate set
    C_i. The state index is the candidate's row in the observation's candidate
    list, so the candidate -> road-segment correspondence is preserved.

Emission  log P(z_i | c)
    w_e * log( p_c / sum_{c' in C_i} p_c' )
    where p_c is the classifier probability for the (observation, candidate)
    row. The classifier already combines distance, heading, context, etc.

Transition  log P(c_i | c_{i-1})
    Newson & Krumm (2009): the network distance travelled between the two
    projected positions should be close to the straight-line distance between
    the two GPS points:
        log P = -|d_route - d_gc| / beta
    d_route = remaining length on arc a + shortest path(end(a) -> start(b))
              + distance along arc b; on the same arc it is the difference of
              positions (small backward moves <= BACK_SLACK_M are tolerated as
              GPS noise). Unreachable pairs (no path within the search limit)
              get log P = -inf.

Gaps / breaks
    * observations with no candidate get no state (predicted arc = -1) and the
      chain bridges over them;
    * if dt > GAP_SECONDS the chain is split and each part decoded separately;
    * if no transition from step i-1 to step i is feasible (disconnected
      segments, e.g. GPS jump or missing road), the chain is broken and
      restarted at i from emissions only. Breaks are counted and reported.

Decoding
    Standard Viterbi in log-space:
        V_i(b) = E_i(b) + max_a [ V_{i-1}(a) + T(a,b) ],  back-pointer = argmax
    followed by back-tracking from argmax_b V_last(b).
"""
from dataclasses import dataclass

import numpy as np

GAP_SECONDS = 60.0
BACK_SLACK_M = 15.0


@dataclass
class HMMParams:
    beta: float = 10.0          # transition scale (m)
    emission_weight: float = 1.0
    min_prob: float = 1e-6      # floor for normalised candidate probabilities
    max_route_factor: float = 4.0   # search limit = max(300, factor * d_gc + 100) m


def _group_candidates(cand_df, proba, n_obs, params):
    """Per observation: arrays of arcs, positions t and log-emissions."""
    obs = cand_df["obs"].values
    order = np.argsort(obs, kind="stable")
    obs_s = obs[order]
    arcs = cand_df["arc"].values[order]
    tpos = cand_df["t_pos"].values[order]
    p = np.maximum(np.asarray(proba)[order], params.min_prob)
    bounds = np.searchsorted(obs_s, np.arange(n_obs + 1))
    out = []
    for i in range(n_obs):
        s, e = bounds[i], bounds[i + 1]
        if e <= s:
            out.append(None)
            continue
        pi = p[s:e] / p[s:e].sum()
        out.append((arcs[s:e], tpos[s:e], params.emission_weight * np.log(np.maximum(pi, params.min_prob))))
    return out


def _transition_matrix(net, prev, cur, d_gc, params):
    a_arcs, a_t, _ = prev
    b_arcs, b_t, _ = cur
    La = net.length[a_arcs]
    Lb = net.length[b_arcs]
    limit = max(300.0, params.max_route_factor * d_gc + 100.0)
    limit = float(np.ceil(limit / 100.0) * 100.0)        # quantise -> better cache reuse
    arcs_tab = net.trip.arcs
    T = np.full((len(a_arcs), len(b_arcs)), -np.inf)
    for ia, a in enumerate(a_arcs):
        rem = (1.0 - a_t[ia]) * La[ia]
        dist_from_end = None
        for ib, b in enumerate(b_arcs):
            if a == b:
                fwd = (b_t[ib] - a_t[ia]) * La[ia]
                if fwd >= -BACK_SLACK_M:
                    d = abs(fwd)
                    T[ia, ib] = -abs(d - d_gc) / params.beta
                    continue
            if dist_from_end is None:
                dist_from_end = net.distances_from(arcs_tab[a, 1], limit)
            sp = dist_from_end.get(int(arcs_tab[b, 0]))
            if sp is None:
                continue
            d = rem + sp + b_t[ib] * Lb[ib]
            T[ia, ib] = -abs(d - d_gc) / params.beta
    return T


def viterbi(trip, net, cand_df, proba, params: HMMParams = HMMParams()):
    """Decode the most likely arc for every observation.

    Returns dict with 'arcs' (per-observation arc id, -1 if none),
    'n_breaks', 'n_segments'."""
    n = len(trip.xy)
    groups = _group_candidates(cand_df, proba, n, params)
    xy, t = trip.xy, trip.times
    result = np.full(n, -1, dtype=np.int64)
    n_breaks = 0
    n_segments = 0

    seg_obs, seg_back = [], []   # observations of current chain and back-pointers
    V = None
    last_i = None

    def flush():
        nonlocal seg_obs, seg_back, V
        if not seg_obs:
            return
        k = int(np.argmax(V))
        for j in range(len(seg_obs) - 1, -1, -1):
            i = seg_obs[j]
            result[i] = groups[i][0][k]
            if j > 0:
                k = seg_back[j][k]
        seg_obs, seg_back, V = [], [], None

    for i in range(n):
        g = groups[i]
        if g is None:
            continue
        new_chain = V is None or (t[i] - t[last_i]) > GAP_SECONDS
        if not new_chain:
            d_gc = float(np.linalg.norm(xy[i] - xy[last_i]))
            T = _transition_matrix(net, groups[last_i], g, d_gc, params)
            S = V[:, None] + T
            best = np.max(S, axis=0)
            if not np.any(np.isfinite(best)):
                n_breaks += 1
                new_chain = True
            else:
                bp = np.argmax(S, axis=0)
                V = best + g[2]
                seg_obs.append(i)
                seg_back.append(bp)
        if new_chain:
            flush()
            n_segments += 1
            V = g[2].copy()
            seg_obs.append(i)
            seg_back.append(None)
        last_i = i
    flush()
    return {"arcs": result, "n_breaks": n_breaks, "n_segments": n_segments}


def reconstruct_route(net, obs_arcs):
    """Turn per-observation arcs into a continuous arc sequence by inserting the
    shortest connecting path between consecutive (different, non-adjacent) arcs."""
    seq = [int(a) for a in obs_arcs if a >= 0]
    seq = [a for i, a in enumerate(seq) if i == 0 or a != seq[i - 1]]
    arcs_tab = net.trip.arcs
    route = []
    for i, a in enumerate(seq):
        if route:
            prev = route[-1]
            u, v = int(arcs_tab[prev, 1]), int(arcs_tab[a, 0])
            if u != v:
                nodes = net.shortest_path_nodes(u, v, limit=5000.0)
                if nodes is not None:
                    for x, y in zip(nodes[:-1], nodes[1:]):
                        e = net.uv_to_arc.get((x, y))
                        if e is not None:
                            route.append(e)
        route.append(a)
    return route

In [ ]:
%%writefile /kaggle/working/PathSnap/pathsnap/pipeline.py
"""End-to-end inference pipeline (used by the API, the map scripts and the HMM experiments).

predict_trip() never reads the .route file: the trip is loaded with
with_route=False, so reference labels cannot enter the prediction.
Scoring against the reference is a separate, server-side step (score_trip).
"""
import json
import time
from pathlib import Path

import joblib
import numpy as np

from . import config, data_io, features, hmm, labels, metrics, network


class PathSnapModel:
    def __init__(self, models_dir=config.MODELS_DIR):
        models_dir = Path(models_dir)
        self.clf = joblib.load(models_dir / "final_model.joblib")
        self.cfg = json.load(open(models_dir / "model_config.json"))
        hp = json.load(open(models_dir / "hmm_config.json")) if (models_dir / "hmm_config.json").exists() else {}
        self.hmm_params = hmm.HMMParams(**hp.get("params", {}))
        self.features = self.cfg["features"]

    def predict_trip(self, partition_dir, trip_id):
        timings = {}
        t0 = time.perf_counter()
        trip = data_io.load_trip(partition_dir, trip_id, with_route=False)
        net = network.RoadNetwork(trip)
        timings["load_and_index_s"] = time.perf_counter() - t0
        t0 = time.perf_counter()
        cand = features.build_candidate_table(trip, net, labels=None,
                                              radius=self.cfg["candidate_radius_m"], k=self.cfg["max_candidates"])
        timings["features_s"] = time.perf_counter() - t0
        t0 = time.perf_counter()
        proba = self.clf.predict_proba(cand[self.features].values)[:, 1] if len(cand) else np.zeros(0)
        timings["classifier_s"] = time.perf_counter() - t0
        n = len(trip.xy)
        clf_arcs = np.full(n, -1, dtype=np.int64)
        if len(cand):
            mask = metrics.top1_from_scores(cand.obs.values, proba)
            clf_arcs[cand.obs.values[mask]] = cand.arc.values[mask]
        t0 = time.perf_counter()
        dec = hmm.viterbi(trip, net, cand, proba, self.hmm_params)
        route = hmm.reconstruct_route(net, dec["arcs"])
        timings["hmm_viterbi_s"] = time.perf_counter() - t0
        return {"trip": trip, "net": net, "candidates": cand, "proba": proba,
                "classifier_arcs": clf_arcs, "hmm_arcs": dec["arcs"], "route": route,
                "n_breaks": dec["n_breaks"], "n_segments": dec["n_segments"], "timings": timings}


def score_trip(partition_dir, trip_id, pred):
    """Server-side evaluation against the reference route (never sent to the model)."""
    ref_trip = data_io.load_trip(partition_dir, trip_id, with_route=True)
    if ref_trip.route is None:
        return None
    ref_obs, _ = labels.align_route(ref_trip)
    net, cand = pred["net"], pred["candidates"]
    in_cand = np.zeros(len(ref_obs), bool)
    if len(cand):
        hit = cand.arc.values == ref_obs[cand.obs.values]
        in_cand[cand.obs.values[hit]] = True
    out = {
        "n_observations": int(len(ref_obs)),
        "hmm_top1_accuracy": metrics.observation_accuracy(pred["hmm_arcs"], ref_obs),
        "classifier_top1_accuracy": metrics.observation_accuracy(pred["classifier_arcs"], ref_obs),
        "candidate_miss_rate": float(1 - in_cand.mean()),
        "hmm_route_continuity": metrics.route_continuity(pred["hmm_arcs"], ref_trip.arcs),
        "classifier_route_continuity": metrics.route_continuity(pred["classifier_arcs"], ref_trip.arcs),
    }
    out.update(metrics.route_overlap(pred["route"], ref_trip.route, net.length))
    if len(cand):
        y = (cand.arc.values == ref_obs[cand.obs.values]).astype(int)
        mask = metrics.top1_from_scores(cand.obs.values, pred["proba"])
        cm = metrics.candidate_metrics(y, mask, pred["proba"])
        out.update({"classifier_macro_f1": cm["macro_f1"], "classifier_log_loss": cm["log_loss"]})
    return out

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/01_eda.py
"""Step 1 - Dataset exploration (training + student-testing partitions only).

Outputs (results/eda/):
  inventory.csv           one row per trip: files, sizes, counts, extents, gaps, anomalies
  network_stats.csv       road-network size and connectivity per trip
  label_quality.csv       GPS-to-reference-route distance per trip (route alignment)
  data_quality_notes.md   automatically generated summary of issues found
  *.png                   plots
  maps/*.html             Folium maps of selected trips (GPS + road network + reference route)

Run:  python scripts/01_eda.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

from pathsnap import config, data_io, geo, labels, visual

OUT = config.RESULTS_DIR / "eda"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "maps").mkdir(exist_ok=True)


def raw_checks(part_dir, tid):
    """Checks on the RAW files (before cleaning)."""
    d = Path(part_dir) / tid
    files = {ext: d / f"{tid}.{ext}" for ext in ["track", "nodes", "arcs", "route"]}
    raw = np.genfromtxt(files["track"], invalid_raise=False)
    n_raw = len(raw)
    n_nan = int(np.sum(~np.isfinite(raw).all(axis=1)))
    t = raw[:, 2]
    dt = np.diff(t)
    return {
        "trip": tid,
        **{f"has_{e}": f.exists() for e, f in files.items()},
        **{f"kb_{e}": round(f.stat().st_size / 1024, 1) if f.exists() else np.nan for e, f in files.items()},
        "n_obs_raw": n_raw, "n_rows_with_nan": n_nan,
        "n_nonmonotonic_t": int(np.sum(dt <= 0)),
        "duration_s": float(t[-1] - t[0]),
        "dt_1s_frac": float(np.mean(dt == 1)), "dt_max_s": float(dt.max()),
        "n_gaps_gt5s": int(np.sum(dt > 5)), "n_gaps_gt30s": int(np.sum(dt > 30)),
        "lon_min": raw[:, 0].min(), "lon_max": raw[:, 0].max(),
        "lat_min": raw[:, 1].min(), "lat_max": raw[:, 1].max(),
    }


def network_checks(trip):
    V = len(trip.nodes)
    a = trip.arcs
    G = csr_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(V, V))
    n_weak, lab_w = connected_components(G, directed=True, connection="weak")
    n_strong, lab_s = connected_components(G, directed=True, connection="strong")
    deg = np.bincount(a[:, 0], minlength=V) + np.bincount(a[:, 1], minlength=V)
    seg_len = np.linalg.norm(trip.node_xy[a[:, 1]] - trip.node_xy[a[:, 0]], axis=1)
    key = set(map(tuple, a.tolist()))
    twoway = np.mean([(v, u) in key for u, v in a.tolist()])
    out = {
        "trip": trip.trip_id, "n_nodes": V, "n_arcs": len(a),
        "n_isolated_nodes": int(np.sum(deg == 0)),
        "n_weak_components": n_weak,
        "largest_weak_comp_frac": float(np.bincount(lab_w).max() / V),
        "n_strong_components": n_strong,
        "largest_strong_comp_frac": float(np.bincount(lab_s).max() / V),
        "twoway_arc_frac": float(twoway),
        "seg_len_median_m": float(np.median(seg_len)), "seg_len_max_m": float(seg_len.max()),
        "n_self_loops": int(np.sum(a[:, 0] == a[:, 1])),
        "n_duplicate_arcs": int(len(a) - len(key)),
    }
    if trip.route is not None:
        r = trip.route
        out["route_len_arcs"] = len(r)
        out["route_unique_arcs"] = len(np.unique(r))
        out["route_connected_frac"] = float(np.mean(a[r[:-1], 1] == a[r[1:], 0])) if len(r) > 1 else 1.0
        out["route_length_km"] = float(seg_len[r].sum() / 1000)
    return out


def main():
    rows, nets, qual = [], [], []
    for part, pdir in [("train", config.TRAIN_DIR), ("student_test", config.STUDENT_TEST_DIR)]:
        for tid in data_io.list_trips(pdir):
            r = raw_checks(pdir, tid)
            r["partition"] = part
            trip = data_io.load_trip(pdir, tid)
            r["n_obs_clean"] = len(trip.xy)
            step = np.linalg.norm(np.diff(trip.xy, axis=0), axis=1)
            sp = step / np.maximum(np.diff(trip.times), 1e-6)
            r["path_len_km"] = step.sum() / 1000
            r["speed_median_ms"] = float(np.median(sp))
            r["speed_p99_ms"] = float(np.percentile(sp, 99))
            r["n_jumps_gt50ms"] = int(np.sum(sp > 50))
            r["stationary_frac"] = float(np.mean(step < 0.5))
            ext = trip.xy.max(0) - trip.xy.min(0)
            r["extent_ew_km"], r["extent_ns_km"] = ext / 1000
            rows.append(r)
            n = network_checks(trip)
            n["partition"] = part
            nets.append(n)
            lab, d = labels.align_route(trip)
            qual.append({"trip": tid, "partition": part,
                         "label_dist_median_m": float(np.median(d)),
                         "label_dist_p90_m": float(np.percentile(d, 90)),
                         "label_dist_max_m": float(d.max()),
                         "frac_obs_gt50m": float(np.mean(d > 50))})
            np.save(OUT / f"_labeldist_{part}_{tid}.npy", d.astype(np.float32))
            print(f"{part} {tid}: n={len(trip.xy)} label-dist median={np.median(d):.1f} m")

    inv = pd.DataFrame(rows)
    net = pd.DataFrame(nets)
    lq = pd.DataFrame(qual)
    inv.to_csv(OUT / "inventory.csv", index=False)
    net.to_csv(OUT / "network_stats.csv", index=False)
    lq.to_csv(OUT / "label_quality.csv", index=False)

    # ---------- summary tables ----------
    summ = inv.groupby("partition").agg(
        trips=("trip", "count"), obs_total=("n_obs_clean", "sum"),
        obs_min=("n_obs_clean", "min"), obs_median=("n_obs_clean", "median"), obs_max=("n_obs_clean", "max"),
        nan_rows=("n_rows_with_nan", "sum"), nonmonotonic=("n_nonmonotonic_t", "sum"),
        gaps_gt5s=("n_gaps_gt5s", "sum"), gaps_gt30s=("n_gaps_gt30s", "sum"),
        dt_1s_frac_mean=("dt_1s_frac", "mean"))
    summ.to_csv(OUT / "summary_by_partition.csv")
    print(summ)

    # ---------- plots ----------
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for part, g in inv.groupby("partition"):
        ax[0].hist(g["n_obs_clean"], bins=30, alpha=.6, label=part)
        ax[1].hist(g["duration_s"] / 60, bins=30, alpha=.6, label=part)
        ax[2].hist(g["speed_median_ms"] * 3.6, bins=30, alpha=.6, label=part)
    ax[0].set(title="GPS observations per trip", xlabel="observations", ylabel="trips")
    ax[1].set(title="Trip duration", xlabel="minutes")
    ax[2].set(title="Median speed per trip", xlabel="km/h")
    for a in ax:
        a.legend()
    fig.tight_layout(); fig.savefig(OUT / "trip_size_distributions.png", dpi=120); plt.close(fig)

    # sampling interval distribution (train)
    dts = []
    for tid in data_io.list_trips(config.TRAIN_DIR):
        t = np.loadtxt(config.TRAIN_DIR / tid / f"{tid}.track")[:, 2]
        dts.append(np.diff(t))
    dts = np.concatenate(dts)
    fig, ax = plt.subplots(figsize=(7, 4))
    vals, cnts = np.unique(np.clip(dts, 0, 11), return_counts=True)
    ax.bar([str(int(v)) if v < 11 else ">10" for v in vals], cnts)
    ax.set_yscale("log"); ax.set(title=f"Sampling interval (train, {len(dts):,} steps)", xlabel="seconds", ylabel="count (log)")
    fig.tight_layout(); fig.savefig(OUT / "sampling_intervals_train.png", dpi=120); plt.close(fig)

    # GPS-to-reference distance (train) -> justifies candidate radius
    d_all = np.concatenate([np.load(OUT / f"_labeldist_train_{t}.npy")
                            for t in lq[lq.partition == "train"].trip])
    good = lq[(lq.partition == "train") & (lq.label_dist_median_m <= 30)].trip
    d_good = np.concatenate([np.load(OUT / f"_labeldist_train_{t}.npy") for t in good])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(np.clip(d_good, 0, 80), bins=80)
    for q in [90, 99, 99.9]:
        v = np.percentile(d_good, q)
        ax[0].axvline(v, ls="--", c="k"); ax[0].text(v, ax[0].get_ylim()[1] * .8, f" p{q}={v:.0f}m", fontsize=8)
    ax[0].set(title="GPS point -> aligned reference arc distance (train, clean trips)", xlabel="metres")
    ax[1].bar(lq.trip + "\n" + lq.partition.str[:2], lq.label_dist_median_m)
    ax[1].axhline(30, c="r", ls="--", label="30 m quality threshold")
    ax[1].set(title="Median GPS-to-reference distance per trip", ylabel="metres"); ax[1].legend()
    ax[1].set_xticks([])
    fig.tight_layout(); fig.savefig(OUT / "gps_to_reference_distance.png", dpi=120); plt.close(fig)
    pd.Series({f"p{q}": float(np.percentile(d_good, q)) for q in [50, 75, 90, 95, 99, 99.5, 99.9]}).to_csv(
        OUT / "gps_noise_percentiles_train_clean.csv", header=["metres"])

    # geographic distribution of trips
    fig, ax = plt.subplots(figsize=(8, 5))
    for part, g in inv.groupby("partition"):
        ax.scatter((g.lon_min + g.lon_max) / 2, (g.lat_min + g.lat_max) / 2, label=part, alpha=.7)
    ax.set(title="Trip centres (lon/lat)", xlabel="longitude", ylabel="latitude"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "trip_locations.png", dpi=120); plt.close(fig)

    # trajectory + network plots for a few trips
    for part, pdir, tid in [("train", config.TRAIN_DIR, "00000001"),
                            ("train", config.TRAIN_DIR, "00000050"),
                            ("student_test", config.STUDENT_TEST_DIR, "00000000")]:
        trip = data_io.load_trip(pdir, tid)
        fig, ax = plt.subplots(figsize=(8, 8))
        lo, hi = trip.track[:, :2].min(0) - 0.003, trip.track[:, :2].max(0) + 0.003
        a = trip.arcs
        m = np.all((trip.nodes[a[:, 0]] >= lo) & (trip.nodes[a[:, 0]] <= hi), axis=1)
        for u, v in a[m]:
            ax.plot(trip.nodes[[u, v], 0], trip.nodes[[u, v], 1], c="#bbbbbb", lw=.5, zorder=1)
        r = trip.route
        for u, v in a[r]:
            ax.plot(trip.nodes[[u, v], 0], trip.nodes[[u, v], 1], c="green", lw=2.5, zorder=2)
        ax.scatter(trip.track[:, 0], trip.track[:, 1], s=3, c="blue", zorder=3, label="GPS")
        ax.plot([], [], c="green", label="reference route"); ax.plot([], [], c="#bbbbbb", label="road network")
        ax.set(title=f"{part} trip {tid}", xlabel="longitude", ylabel="latitude"); ax.legend()
        ax.set_aspect(1 / np.cos(np.radians(trip.track[:, 1].mean())))
        fig.tight_layout(); fig.savefig(OUT / f"trajectory_{part}_{tid}.png", dpi=120); plt.close(fig)
        visual.trip_map(trip, show_network=True, reference_arcs=trip.route,
                        title=f"{part} trip {tid}: raw GPS, road network and reference route"
                        ).save(str(OUT / "maps" / f"eda_{part}_{tid}.html"))

    # network size plot
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].scatter(net.n_nodes, net.n_arcs, c=(net.partition == "train").map({True: "C0", False: "C1"}))
    ax[0].set(xscale="log", yscale="log", title="Road network size per trip", xlabel="nodes", ylabel="arcs")
    ax[1].hist(net.largest_strong_comp_frac, bins=20)
    ax[1].set(title="Largest strongly-connected component (fraction of nodes)", xlabel="fraction")
    fig.tight_layout(); fig.savefig(OUT / "network_size_connectivity.png", dpi=120); plt.close(fig)

    # ---------- automatic data-quality notes ----------
    bad = lq[lq.label_dist_median_m > 30]
    notes = [
        "# Data-quality notes (auto-generated by scripts/01_eda.py)", "",
        f"* Partitions analysed: train ({(inv.partition == 'train').sum()} trips), "
        f"student_test ({(inv.partition == 'student_test').sum()} trips). Evaluator-testing data was not used.",
        f"* Rows with missing/non-numeric values: {int(inv.n_rows_with_nan.sum())}.",
        f"* Non-increasing timestamps: {int(inv.n_nonmonotonic_t.sum())}.",
        f"* Mean fraction of 1-second steps: {inv.dt_1s_frac.mean():.3f}; gaps >5 s: {int(inv.n_gaps_gt5s.sum())}; gaps >30 s: {int(inv.n_gaps_gt30s.sum())}; largest gap {inv.dt_max_s.max():.0f} s.",
        f"* Trips spread over several countries/continents (lon {inv.lon_min.min():.1f}..{inv.lon_max.max():.1f}, lat {inv.lat_min.min():.1f}..{inv.lat_max.max():.1f}) -> per-trip local projection.",
        f"* Every trip has its own road-network files; arc ids are trip-local.",
        f"* Reference routes are connected arc chains (mean consecutive-connected fraction {net.route_connected_frac.mean():.3f}).",
        f"* Two-way roads are stored as two opposite directed arcs (median two-way fraction {net.twoway_arc_frac.median():.2f}): the nearest segment alone cannot tell the direction -> heading features.",
        f"* GPS->reference distance on clean training trips: p50={np.percentile(d_good, 50):.1f} m, p90={np.percentile(d_good, 90):.1f} m, p99={np.percentile(d_good, 99):.1f} m, p99.9={np.percentile(d_good, 99.9):.1f} m.",
        f"* Trips whose reference route does not follow the GPS (median distance > 30 m): "
        + (", ".join(f"{r.trip} ({r.partition}, {r.label_dist_median_m:.0f} m)" for r in bad.itertuples()) or "none"),
        "  These labels are unreliable; such TRAIN trips are excluded from model fitting (student-test trips are kept and reported as-is).",
    ]
    (OUT / "data_quality_notes.md").write_text("\n".join(notes), encoding="utf-8")
    print("\n".join(notes))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/02_build_features.py
"""Step 2 - Candidate generation + feature engineering for train and student_test.

For every trip:
  1. load + clean + project the track (pathsnap.data_io)
  2. derive observation-level reference arcs from .route (pathsnap.labels)
  3. generate candidate arcs within CANDIDATE_RADIUS_M (pathsnap.network)
  4. compute candidate features (pathsnap.features)

Outputs (results/features/):
  <partition>_candidates.parquet   one row per (trip, obs, candidate arc) with features + y
  <partition>_observations.csv     one row per GPS observation: reference arc, distance, candidate hit
  candidate_stats.csv              candidate miss rate etc. per trip

Run:  python scripts/02_build_features.py
"""
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import pandas as pd

from pathsnap import config, data_io, features, labels, network

OUT = config.FEATURES_DIR
OUT.mkdir(parents=True, exist_ok=True)


def main():
    stats = []
    for part, pdir in [("train", config.TRAIN_DIR), ("student_test", config.STUDENT_TEST_DIR)]:
        tables, obs_rows = [], []
        for tid in data_io.list_trips(pdir):
            t0 = time.time()
            trip = data_io.load_trip(pdir, tid)
            lab, ldist = labels.align_route(trip)
            net = network.RoadNetwork(trip)
            df = features.build_candidate_table(trip, net, lab)
            hit = np.zeros(len(trip.xy), bool)
            hit[df.loc[df.y == 1, "obs"].values] = True
            nearest = df[df.dist_rank == 0]
            obs_rows.append(pd.DataFrame({"trip": tid, "obs": np.arange(len(trip.xy)), "ref_arc": lab,
                                          "ref_dist_m": ldist, "ref_in_candidates": hit}))
            tables.append(df)
            stats.append({"partition": part, "trip": tid, "n_obs": len(trip.xy), "n_rows": len(df),
                          "mean_candidates": len(df) / len(trip.xy),
                          "obs_without_candidates": int(len(trip.xy) - df.obs.nunique()),
                          "candidate_miss_rate": float(1 - hit.mean()),
                          "nearest_arc_top1": float(nearest.y.sum() / len(trip.xy)),
                          "label_dist_median_m": float(np.median(ldist)),
                          "seconds": round(time.time() - t0, 2)})
            print(stats[-1])
        pd.concat(tables).to_parquet(OUT / f"{part}_candidates.parquet", index=False)
        pd.concat(obs_rows).to_csv(OUT / f"{part}_observations.csv", index=False)
    st = pd.DataFrame(stats)
    st.to_csv(OUT / "candidate_stats.csv", index=False)
    print(st.groupby("partition")[["mean_candidates", "candidate_miss_rate", "nearest_arc_top1"]].mean())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/03_train_models.py
"""Step 3 - Train and compare candidate classifiers.

Protocol
  * Fitting / CV: 80 training trips (minus label-unreliable trips found in EDA),
    5-fold GroupKFold by trip (no trip is split across folds).
  * Development comparison: 10 student-testing trips (every observation counts).
  * Evaluator-testing data is never touched.

Outputs (results/models/):
  cv_folds.csv, model_comparison.csv, confusion_<model>.png, comparison plots,
  student_test_proba_<model>.npy (for the HMM experiments)

Run:  python scripts/03_train_models.py                 (all models)
      python scripts/03_train_models.py --models MLP KNN (subset)
"""
import argparse
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from pathsnap import config, training
from pathsnap.features import FEATURES
from pathsnap.models import MODEL_SPECS

OUT = config.RESULTS_DIR / "models"
OUT.mkdir(parents=True, exist_ok=True)


def run_model(name, tr, te, te_obs, arcs_te):
    spec = MODEL_SPECS[name]
    X, y, groups = tr[FEATURES].values, tr.y.values, tr.trip.values
    folds = []
    for k, (i_tr, i_va) in enumerate(GroupKFold(n_splits=config.N_FOLDS).split(X, y, groups)):
        Xs, ys = training.subsample(X[i_tr], y[i_tr], spec["max_rows"])
        m = spec["factory"]()
        t0 = time.perf_counter()
        m.fit(Xs, ys)
        fit_s = time.perf_counter() - t0
        p, _ = training.timed_predict(m, X[i_va])
        r = training.evaluate_rows(tr.iloc[i_va], p)
        r.update(model=name, fold=k, fit_seconds=fit_s, n_train_rows=len(Xs))
        folds.append(r)
        print(f"  {name} fold {k}: macroF1={r['macro_f1']:.4f} top1={r['top1_acc']:.4f}", flush=True)
    # final fit on all training rows
    Xs, ys = training.subsample(X, y, spec["max_rows"])
    m = spec["factory"]()
    t0 = time.perf_counter()
    m.fit(Xs, ys)
    fit_s = time.perf_counter() - t0
    # training-set (in-sample) metrics, for over/under-fitting diagnosis
    p_in, _ = training.timed_predict(m, X)
    r_in = training.evaluate_rows(tr, p_in)
    p, inf_s = training.timed_predict(m, te[FEATURES].values)
    res, preds = training.evaluate_partition(te, te_obs, p, arcs_te)
    np.save(OUT / f"student_test_proba_{name}.npy", p.astype(np.float32))
    cv = pd.DataFrame(folds)
    summary = {
        "model": name, "family": spec["family"], "config": spec["config"],
        "cv_macro_f1_mean": cv.macro_f1.mean(), "cv_macro_f1_std": cv.macro_f1.std(),
        "cv_top1_mean": cv.top1_acc.mean(), "cv_log_loss_mean": cv.log_loss.mean(),
        "train_macro_f1": r_in["macro_f1"], "train_top1": r_in["top1_acc"],
        "test_top1": res["top1_acc"], "test_macro_f1": res["macro_f1"],
        "test_precision": res["precision"], "test_recall": res["recall"], "test_f1": res["f1"],
        "test_log_loss": res["log_loss"], "test_route_continuity": res["route_continuity"],
        "fit_seconds": fit_s, "n_train_rows": len(Xs),
        "inference_ms_per_1k_obs": 1000 * inf_s / te_obs.shape[0] * 1000,
    }
    return summary, cv, p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", default=list(MODEL_SPECS))
    args = ap.parse_args()
    tr = training.load_training_rows()
    te, te_obs = training.load_student_test()
    arcs_te = training.arcs_tables(config.STUDENT_TEST_DIR, sorted(te_obs.trip.unique()))
    print(f"training rows={len(tr):,} obs={tr.groupby(['trip','obs']).ngroups:,} trips={tr.trip.nunique()}; "
          f"student-test rows={len(te):,} obs={len(te_obs):,}")

    # non-learned reference: nearest segment
    base, _ = training.evaluate_partition(te, te_obs, -te.dist.values, arcs_te)
    json.dump(base, open(OUT / "baseline_nearest_segment.json", "w"), indent=2)
    print("nearest-segment baseline (student test):", base)

    for name in args.models:
        print(f"== {name}", flush=True)
        summary, cv, _ = run_model(name, tr, te, te_obs, arcs_te)
        cv.to_csv(OUT / f"cv_folds_{name}.csv", index=False)
        pd.Series(summary).to_json(OUT / f"summary_{name}.json", indent=2)
        print(pd.Series(summary).to_string(), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/03b_learning_curves.py
"""Step 3b - Epoch/iteration-wise training histories (only for models that have them).

* HistGradientBoosting: log loss after every boosting iteration (staged predictions)
* MLP: log loss after every epoch (partial_fit loop)

Training-partition trips are split by trip into 80 % fit / 20 % validation
(GroupShuffleSplit), so the validation curve is on unseen trips.
Other models (LR, NB, KNN, DT, RF) do not produce epoch-wise histories, so no
curves are drawn for them.

Run:  python scripts/03b_learning_curves.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from pathsnap import config, training
from pathsnap.features import FEATURES

OUT = config.RESULTS_DIR / "models"


def main():
    tr = training.load_training_rows()
    X, y, g = tr[FEATURES].values, tr.y.values, tr.trip.values
    i_fit, i_val = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=config.RANDOM_SEED).split(X, y, g))
    Xf, yf, Xv, yv = X[i_fit], y[i_fit], X[i_val], y[i_val]
    rng = np.random.default_rng(config.RANDOM_SEED)
    sub = rng.choice(len(Xf), min(200_000, len(Xf)), replace=False)   # train-loss estimate sample

    # ---- HistGradientBoosting ----
    hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31,
                                         early_stopping=False, random_state=config.RANDOM_SEED).fit(Xf, yf)
    tr_l = [log_loss(yf[sub], p[:, 1], labels=[0, 1]) for p in hgb.staged_predict_proba(Xf[sub])]
    va_l = [log_loss(yv, p[:, 1], labels=[0, 1]) for p in hgb.staged_predict_proba(Xv)]
    h1 = pd.DataFrame({"iteration": np.arange(1, len(tr_l) + 1), "train_log_loss": tr_l, "val_log_loss": va_l})
    h1.to_csv(OUT / "history_HistGradientBoosting.csv", index=False)

    # ---- MLP ----
    sc = StandardScaler().fit(Xf)
    Xfs, Xvs = sc.transform(Xf), sc.transform(Xv)
    msub = rng.choice(len(Xfs), min(300_000, len(Xfs)), replace=False)
    mlp = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, batch_size=1024,
                        learning_rate_init=1e-3, random_state=config.RANDOM_SEED)
    rows = []
    for ep in range(1, 31):
        idx = rng.permutation(msub)
        mlp.partial_fit(Xfs[idx], yf[idx], classes=[0, 1])
        rows.append({"epoch": ep,
                     "train_log_loss": log_loss(yf[sub], mlp.predict_proba(Xfs[sub])[:, 1], labels=[0, 1]),
                     "val_log_loss": log_loss(yv, mlp.predict_proba(Xvs)[:, 1], labels=[0, 1])})
        print(rows[-1], flush=True)
    h2 = pd.DataFrame(rows)
    h2.to_csv(OUT / "history_MLP.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h1.iteration, h1.train_log_loss, label="train (fit trips)")
    ax[0].plot(h1.iteration, h1.val_log_loss, label="validation (held-out trips)")
    ax[0].set(title="HistGradientBoosting: log loss per boosting iteration", xlabel="iteration", ylabel="log loss")
    ax[1].plot(h2.epoch, h2.train_log_loss, label="train (fit trips)")
    ax[1].plot(h2.epoch, h2.val_log_loss, label="validation (held-out trips)")
    ax[1].set(title="MLP (64,32): log loss per epoch", xlabel="epoch", ylabel="log loss")
    for a in ax:
        a.legend(); a.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUT / "training_histories.png", dpi=120)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/03c_compare_models.py
"""Step 3c - Consolidated model comparison table + plots (from saved step-3 results).

Outputs (results/models/): model_comparison.csv, cv_macro_f1_comparison.png,
student_test_comparison.png, log_loss_comparison.png, efficiency.png,
confusion_matrices.png

Run:  python scripts/03c_compare_models.py
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

from pathsnap import config, metrics, training
from pathsnap.models import MODEL_SPECS

OUT = config.RESULTS_DIR / "models"


def main():
    names = [n for n in MODEL_SPECS if (OUT / f"summary_{n}.json").exists()]
    summ = pd.DataFrame([json.load(open(OUT / f"summary_{n}.json")) for n in names])
    folds = pd.concat([pd.read_csv(OUT / f"cv_folds_{n}.csv") for n in names])
    base = json.load(open(OUT / "baseline_nearest_segment.json"))
    summ.to_csv(OUT / "model_comparison.csv", index=False)
    cols = ["model", "family", "cv_macro_f1_mean", "cv_macro_f1_std", "train_macro_f1", "test_top1",
            "test_macro_f1", "test_log_loss", "test_route_continuity", "fit_seconds", "inference_ms_per_1k_obs"]
    print(summ[cols].round(4).to_string(index=False))
    print("nearest-segment baseline:", {k: round(v, 4) for k, v in base.items()})

    order = summ.sort_values("cv_macro_f1_mean").model.tolist()
    s = summ.set_index("model").loc[order]

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].barh(order, s.cv_macro_f1_mean, xerr=s.cv_macro_f1_std, capsize=3, color="C0")
    lo = s.cv_macro_f1_mean.min() - 0.03
    ax[0].set_xlim(lo, 1.0)
    for i, v in enumerate(s.cv_macro_f1_mean):
        ax[0].text(0.998, i, f"{v:.4f}", va="center", ha="right", fontsize=8)
    ax[0].set(title="5-fold GroupKFold CV Macro-F1 (mean +/- std)", xlabel="Macro-F1")
    data = [folds[folds.model == n].macro_f1.values for n in order]
    try:                                   # matplotlib >= 3.10
        ax[1].boxplot(data, orientation="horizontal")
    except TypeError:                      # older matplotlib (e.g. on Kaggle)
        ax[1].boxplot(data, vert=False)
    ax[1].set_yticks(range(1, len(order) + 1), order)
    ax[1].set(title="CV Macro-F1 per fold (stability)", xlabel="Macro-F1")
    fig.tight_layout(); fig.savefig(OUT / "cv_macro_f1_comparison.png", dpi=120); plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    x = np.arange(len(order))
    ax.bar(x - .2, s.test_top1, .4, label="Top-1 accuracy")
    ax.bar(x + .2, s.test_macro_f1, .4, label="Macro-F1")
    ax.axhline(base["top1_acc"], ls="--", c="gray", label=f"nearest-segment baseline top-1 ({base['top1_acc']:.3f})")
    ax.set_xticks(x, order, rotation=20); ax.set_ylim(0.5, 1.0)
    ax.set(title="Student-testing partition (10 trips): classifier-only", ylabel="score"); ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(OUT / "student_test_comparison.png", dpi=120); plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(order, s.test_log_loss, color="C3")
    ax.set_yscale("log"); ax.set(title="Student-test log loss (lower is better, log scale)", ylabel="log loss")
    for i, v in enumerate(s.test_log_loss):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    plt.xticks(rotation=20); fig.tight_layout(); fig.savefig(OUT / "log_loss_comparison.png", dpi=120); plt.close(fig)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].bar(order, s.fit_seconds); ax[0].set(title="Training time (final fit)", ylabel="seconds")
    ax[1].bar(order, s.inference_ms_per_1k_obs); ax[1].set(title="Inference time", ylabel="ms per 1000 GPS obs")
    ax[1].set_yscale("log")
    for a in ax:
        a.tick_params(axis="x", rotation=30)
    fig.tight_layout(); fig.savefig(OUT / "efficiency.png", dpi=120); plt.close(fig)

    te, _ = training.load_student_test()
    fig, axes = plt.subplots(2, int(np.ceil(len(names) / 2)), figsize=(4 * int(np.ceil(len(names) / 2)), 7))
    for a, n in zip(axes.ravel(), names):
        p = np.load(OUT / f"student_test_proba_{n}.npy")
        key = te.trip + "_" + te.obs.astype(str)
        mask = metrics.top1_from_scores(key.values, p)
        cm = confusion_matrix(te.y.values, mask.astype(int), labels=[0, 1])
        a.imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                       color="white" if cm[i, j] > cm.max() / 2 else "black")
        a.set_xticks([0, 1], ["not chosen", "chosen (top-1)"]); a.set_yticks([0, 1], ["wrong arc", "true arc"])
        a.set_title(n, fontsize=10)
    for a in axes.ravel()[len(names):]:
        a.axis("off")
    fig.suptitle("Candidate-level confusion matrices (student test)")
    fig.tight_layout(); fig.savefig(OUT / "confusion_matrices.png", dpi=110); plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/04_bayes_opt.py
"""Step 4 - Bayesian hyperparameter optimisation of the selected model.

Selected model: HistGradientBoostingClassifier (see results/models/model_comparison.csv
and docs/ for the reasoning).

Method   : Optuna TPE sampler (Tree-structured Parzen Estimator = sequential
           model-based / Bayesian optimisation), seed 42.
Objective: maximise mean group-aware CV Macro-F1 (3-fold GroupKFold by trip on
           the training partition). Student-test and evaluator data are NOT used.
Search space (justification in docs/):
    learning_rate      log-uniform [0.02, 0.3]
    max_iter           int [100, 600]
    max_leaf_nodes     int [15, 127]
    min_samples_leaf   log-int [20, 1000]
    l2_regularization  log-uniform [1e-4, 10]
    max_features       uniform [0.5, 1.0]   (feature subsampling per split)

Outputs (results/bayes_opt/): trials.csv (full search record), best_params.json,
tuned_vs_untuned.csv, optimisation_history.png, param_importance.png
and the final model in models/.

Run:  python scripts/04_bayes_opt.py --trials 15
"""
import argparse
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

from pathsnap import config, training
from pathsnap.features import FEATURES

OUT = config.RESULTS_DIR / "bayes_opt"
OUT.mkdir(parents=True, exist_ok=True)
config.MODELS_DIR.mkdir(exist_ok=True)

UNTUNED = dict(max_iter=300, learning_rate=0.1, max_leaf_nodes=31, early_stopping=False,
               random_state=config.RANDOM_SEED)


def cv_score(params, tr, n_folds):
    X, y, g = tr[FEATURES].values, tr.y.values, tr.trip.values
    scores, tops, lls = [], [], []
    for i_tr, i_va in GroupKFold(n_splits=n_folds).split(X, y, g):
        m = HistGradientBoostingClassifier(**params).fit(X[i_tr], y[i_tr])
        r = training.evaluate_rows(tr.iloc[i_va], m.predict_proba(X[i_va])[:, 1])
        scores.append(r["macro_f1"]); tops.append(r["top1_acc"]); lls.append(r["log_loss"])
    return float(np.mean(scores)), float(np.mean(tops)), float(np.mean(lls))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--trials", type=int, default=15)
    ap.add_argument("--folds", type=int, default=3)
    args = ap.parse_args()
    tr = training.load_training_rows()
    te, te_obs = training.load_student_test()
    arcs_te = training.arcs_tables(config.STUDENT_TEST_DIR, sorted(te_obs.trip.unique()))

    def objective(trial):
        params = dict(
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
            max_iter=trial.suggest_int("max_iter", 100, 600, step=50),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 127),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 20, 1000, log=True),
            l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
            max_features=trial.suggest_float("max_features", 0.5, 1.0),
            early_stopping=False, random_state=config.RANDOM_SEED)
        t0 = time.time()
        f1, top1, ll = cv_score(params, tr, args.folds)
        trial.set_user_attr("cv_top1", top1)
        trial.set_user_attr("cv_log_loss", ll)
        trial.set_user_attr("seconds", time.time() - t0)
        print(f"trial {trial.number}: macroF1={f1:.5f} top1={top1:.4f} ({time.time()-t0:.0f}s) {trial.params}", flush=True)
        return f1

    storage = f"sqlite:///{(OUT / 'optuna_study.db').as_posix()}"
    if (OUT / "optuna_study.db").exists():
        (OUT / "optuna_study.db").unlink()
    study = optuna.create_study(direction="maximize", study_name="pathsnap_hgb", storage=storage,
                                sampler=optuna.samplers.TPESampler(seed=config.RANDOM_SEED, n_startup_trials=5))
    # trial 0 = the untuned configuration, so the search starts from the baseline
    study.enqueue_trial(dict(learning_rate=0.1, max_iter=300, max_leaf_nodes=31, min_samples_leaf=20,
                             l2_regularization=1e-4, max_features=1.0))
    t_search = time.time()
    study.optimize(objective, n_trials=args.trials)
    search_s = time.time() - t_search
    df = study.trials_dataframe()
    df.to_csv(OUT / "trials.csv", index=False)
    best = study.best_params
    json.dump({"best_params": best, "best_cv_macro_f1": study.best_value, "n_trials": len(study.trials),
               "cv_folds": args.folds, "search_seconds": search_s, "sampler": "TPE (seed 42)"},
              open(OUT / "best_params.json", "w"), indent=2)

    # ---- tuned vs untuned, same protocol (5-fold CV + student test) ----
    rows = []
    for name, params in [("untuned", UNTUNED),
                         ("tuned", dict(best, early_stopping=False, random_state=config.RANDOM_SEED))]:
        f1, top1, ll = cv_score(params, tr, config.N_FOLDS)
        t0 = time.time()
        m = HistGradientBoostingClassifier(**params).fit(tr[FEATURES].values, tr.y.values)
        fit_s = time.time() - t0
        p, inf_s = training.timed_predict(m, te[FEATURES].values)
        res, _ = training.evaluate_partition(te, te_obs, p, arcs_te)
        np.save(OUT / f"student_test_proba_{name}.npy", p.astype(np.float32))
        rows.append({"model": f"HGB_{name}", "cv_macro_f1_5fold": f1, "cv_top1_5fold": top1, "cv_log_loss": ll,
                     "test_top1": res["top1_acc"], "test_macro_f1": res["macro_f1"],
                     "test_log_loss": res["log_loss"], "test_route_continuity": res["route_continuity"],
                     "fit_seconds": fit_s, "inference_ms_per_1k_obs": 1e6 * inf_s / len(te_obs)})
        if name == "tuned":
            joblib.dump(m, config.MODELS_DIR / "final_model.joblib", compress=3)
            json.dump({"model": "HistGradientBoostingClassifier", "params": params, "features": FEATURES,
                       "candidate_radius_m": config.CANDIDATE_RADIUS_M, "max_candidates": config.MAX_CANDIDATES},
                      open(config.MODELS_DIR / "model_config.json", "w"), indent=2)
        print(rows[-1], flush=True)
    pd.DataFrame(rows).to_csv(OUT / "tuned_vs_untuned.csv", index=False)

    # ---- plots ----
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df.number, df.value, "o", label="trial CV Macro-F1")
    ax.plot(df.number, df.value.cummax(), "-", label="best so far")
    ax.set(title=f"Bayesian optimisation (TPE) history, {len(df)} trials", xlabel="trial", ylabel="3-fold CV Macro-F1")
    ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUT / "optimisation_history.png", dpi=120); plt.close(fig)
    try:
        imp = optuna.importance.get_param_importances(study)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.barh(list(imp.keys())[::-1], list(imp.values())[::-1])
        ax.set(title="Hyperparameter importance (fANOVA)", xlabel="importance")
        fig.tight_layout(); fig.savefig(OUT / "param_importance.png", dpi=120); plt.close(fig)
        pd.Series(imp).to_csv(OUT / "param_importance.csv", header=["importance"])
    except Exception as e:  # importance needs >1 distinct value per param
        print("importance skipped:", e)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/05_hmm_decoding.py
"""Step 5 - HMM/Viterbi decoding experiments.

1. Decoder-parameter tuning (beta, emission_weight) on TRAINING trips using
   out-of-fold classifier probabilities (5-fold GroupKFold, the held-out trip's
   probabilities come from a model that never saw it). Grid search; objective =
   observation-level Top-1 accuracy after decoding.
2. Student-test comparison: classifier-only vs classifier + HMM/Viterbi, for
   the final tuned model AND for every model from step 3 (decoder compatibility).
3. Per-trip analysis (where HMM helped / did not help).

Outputs (results/hmm/) + models/hmm_config.json

Run:  python scripts/05_hmm_decoding.py
"""
import itertools
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

from pathsnap import config, data_io, hmm, metrics, network, training
from pathsnap.features import FEATURES

OUT = config.RESULTS_DIR / "hmm"
OUT.mkdir(parents=True, exist_ok=True)
N_TUNE_TRIPS = 30


def decode_eval(part_dir, cands, proba, obs_table, params, trips, cache):
    rows = []
    for tid in trips:
        m = (cands.trip == tid).values
        df, p = cands[m], proba[m]
        if tid not in cache:
            trip = data_io.load_trip(part_dir, tid)
            cache[tid] = (trip, network.RoadNetwork(trip))
        trip, net = cache[tid]
        ref = obs_table[obs_table.trip == tid].ref_arc.values
        clf = np.full(len(ref), -1)
        if len(df):
            mask = metrics.top1_from_scores(df.obs.values, p)
            clf[df.obs.values[mask]] = df.arc.values[mask]
        t0 = time.perf_counter()
        r = hmm.viterbi(trip, net, df, p, params)
        route = hmm.reconstruct_route(net, r["arcs"])
        dt = time.perf_counter() - t0
        ov_h = metrics.route_overlap(route, trip.route, net.length)
        ov_c = metrics.route_overlap(hmm.reconstruct_route(net, clf), trip.route, net.length)
        rows.append({"trip": tid, "n_obs": len(ref),
                     "clf_top1": metrics.observation_accuracy(clf, ref),
                     "hmm_top1": metrics.observation_accuracy(r["arcs"], ref),
                     "clf_continuity": metrics.route_continuity(clf, trip.arcs),
                     "hmm_continuity": metrics.route_continuity(r["arcs"], trip.arcs),
                     "clf_route_f1": ov_c["route_f1"], "hmm_route_f1": ov_h["route_f1"],
                     "hmm_route_precision": ov_h["route_precision"], "hmm_route_recall": ov_h["route_recall"],
                     "n_breaks": r["n_breaks"], "n_segments": r["n_segments"], "decode_seconds": dt})
    return pd.DataFrame(rows)


def weighted(df, col):
    return float(np.average(df[col], weights=df.n_obs))


def main():
    final_cfg = json.load(open(config.MODELS_DIR / "model_config.json"))
    params = final_cfg["params"]

    # ---------- 1. out-of-fold probabilities on training trips ----------
    all_tr = pd.read_parquet(config.FEATURES_DIR / "train_candidates.parquet")
    tr_obs = pd.read_csv(config.FEATURES_DIR / "train_observations.csv", dtype={"trip": str})
    ok = sorted(training.label_ok_trips())
    all_tr = all_tr[all_tr.trip.isin(ok)].reset_index(drop=True)
    fit_rows = training.load_training_rows()
    oof = np.zeros(len(all_tr))
    trips_arr = np.array(ok)
    for k, (i_a, i_b) in enumerate(GroupKFold(n_splits=config.N_FOLDS).split(trips_arr, groups=trips_arr)):
        fit_trips, held = set(trips_arr[i_a]), set(trips_arr[i_b])
        f = fit_rows[fit_rows.trip.isin(fit_trips)]
        m = HistGradientBoostingClassifier(**params).fit(f[FEATURES].values, f.y.values)
        sel = all_tr.trip.isin(held).values
        oof[sel] = m.predict_proba(all_tr.loc[sel, FEATURES].values)[:, 1]
        print("oof fold", k, flush=True)

    rng = np.random.default_rng(config.RANDOM_SEED)
    tune_trips = sorted(rng.choice(ok, size=min(N_TUNE_TRIPS, len(ok)), replace=False).tolist())
    cache = {}
    grid = []
    for beta, w in itertools.product([2.0, 5.0, 10.0, 20.0, 40.0], [0.5, 1.0, 2.0]):
        hp = hmm.HMMParams(beta=beta, emission_weight=w)
        res = decode_eval(config.TRAIN_DIR, all_tr, oof, tr_obs, hp, tune_trips, cache)
        grid.append({"beta": beta, "emission_weight": w, "hmm_top1": weighted(res, "hmm_top1"),
                     "clf_top1": weighted(res, "clf_top1"), "hmm_continuity": weighted(res, "hmm_continuity"),
                     "hmm_route_f1": res.hmm_route_f1.mean()})
        print(grid[-1], flush=True)
    grid = pd.DataFrame(grid)
    grid.to_csv(OUT / "decoder_param_search_train_oof.csv", index=False)
    best = grid.loc[grid.hmm_top1.idxmax()]
    best_params = {"beta": float(best.beta), "emission_weight": float(best.emission_weight)}
    json.dump({"params": best_params, "tuned_on": f"{len(tune_trips)} training trips, out-of-fold probabilities",
               "objective": "observation-level top-1 accuracy", "gap_seconds": hmm.GAP_SECONDS,
               "back_slack_m": hmm.BACK_SLACK_M},
              open(config.MODELS_DIR / "hmm_config.json", "w"), indent=2)
    print("best decoder params:", best_params)

    # heat-map of the search
    piv = grid.pivot(index="beta", columns="emission_weight", values="hmm_top1")
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(piv.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(piv.columns)), piv.columns); ax.set_yticks(range(len(piv.index)), piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i, j]:.4f}", ha="center", va="center", color="w", fontsize=8)
    ax.set(xlabel="emission weight", ylabel="beta (m)", title="HMM top-1 (train OOF) vs decoder params")
    fig.colorbar(im); fig.tight_layout(); fig.savefig(OUT / "decoder_param_search.png", dpi=120); plt.close(fig)

    # ---------- 2. student test ----------
    te, te_obs = training.load_student_test()
    hp = hmm.HMMParams(**best_params)
    st_trips = sorted(te_obs.trip.unique())
    cache = {}
    final = joblib.load(config.MODELS_DIR / "final_model.joblib")
    p_final = final.predict_proba(te[FEATURES].values)[:, 1]
    per_trip = decode_eval(config.STUDENT_TEST_DIR, te, p_final, te_obs, hp, st_trips, cache)
    per_trip.to_csv(OUT / "student_test_per_trip_final_model.csv", index=False)
    summ = [{"model": "HGB_tuned (final)", **{c: weighted(per_trip, c) for c in
                                              ["clf_top1", "hmm_top1", "clf_continuity", "hmm_continuity"]},
             "clf_route_f1": per_trip.clf_route_f1.mean(), "hmm_route_f1": per_trip.hmm_route_f1.mean(),
             "decode_seconds_total": per_trip.decode_seconds.sum()}]
    for f in sorted((config.RESULTS_DIR / "models").glob("student_test_proba_*.npy")):
        name = f.stem.replace("student_test_proba_", "")
        r = decode_eval(config.STUDENT_TEST_DIR, te, np.load(f).astype(float), te_obs, hp, st_trips, cache)
        summ.append({"model": name, **{c: weighted(r, c) for c in
                                       ["clf_top1", "hmm_top1", "clf_continuity", "hmm_continuity"]},
                     "clf_route_f1": r.clf_route_f1.mean(), "hmm_route_f1": r.hmm_route_f1.mean(),
                     "decode_seconds_total": r.decode_seconds.sum()})
        print(summ[-1], flush=True)
    summ = pd.DataFrame(summ)
    summ.to_csv(OUT / "classifier_vs_hmm_student_test.csv", index=False)

    # ---------- 3. plots ----------
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    x = np.arange(len(summ))
    ax[0].bar(x - .2, summ.clf_top1, .4, label="classifier only")
    ax[0].bar(x + .2, summ.hmm_top1, .4, label="classifier + HMM/Viterbi")
    ax[0].set_xticks(x, summ.model, rotation=30, ha="right"); ax[0].set_ylim(0.5, 1)
    ax[0].set(title="Student-test observation Top-1 accuracy", ylabel="accuracy"); ax[0].legend()
    ax[1].bar(x - .2, summ.clf_continuity, .4, label="classifier only")
    ax[1].bar(x + .2, summ.hmm_continuity, .4, label="classifier + HMM/Viterbi")
    ax[1].set_xticks(x, summ.model, rotation=30, ha="right"); ax[1].set_ylim(0.5, 1)
    ax[1].set(title="Student-test route continuity", ylabel="fraction of valid transitions"); ax[1].legend()
    fig.tight_layout(); fig.savefig(OUT / "classifier_vs_hmm.png", dpi=120); plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(per_trip))
    ax.bar(x - .2, per_trip.clf_top1, .4, label="classifier only")
    ax.bar(x + .2, per_trip.hmm_top1, .4, label="+ HMM/Viterbi")
    ax.set_xticks(x, per_trip.trip, rotation=45); ax.set_ylim(0.4, 1)
    ax.set(title="Per student-test trip Top-1 accuracy (final model)", ylabel="accuracy"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "per_trip_student_test.png", dpi=120); plt.close(fig)
    print(summ.to_string())
    print(per_trip.to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/06_make_maps.py
"""Step 6 - Folium maps of the final pipeline on student-testing trips.

Each map has layers (toggle with the layer control): raw GPS observations,
road network near the trajectory, reference route, classifier-only
prediction and the HMM/Viterbi predicted route, plus a legend.
Reference routes are drawn here for the REPORT only; the API/UI never send them.

Run:  python scripts/06_make_maps.py              (all 10 student-test trips)
      python scripts/06_make_maps.py 00000000     (one trip)
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import pandas as pd

from pathsnap import config, data_io, visual
from pathsnap.pipeline import PathSnapModel, score_trip

OUT = config.RESULTS_DIR / "maps"
OUT.mkdir(parents=True, exist_ok=True)


def main():
    model = PathSnapModel()
    trips = sys.argv[1:] or data_io.list_trips(config.STUDENT_TEST_DIR)
    rows = []
    for tid in trips:
        pred = model.predict_trip(config.STUDENT_TEST_DIR, tid)
        sc = score_trip(config.STUDENT_TEST_DIR, tid, pred)
        ref = data_io.load_trip(config.STUDENT_TEST_DIR, tid)
        trip = pred["trip"]
        m = visual.trip_map(trip, show_network=True, reference_arcs=ref.route,
                            predicted_arcs=pred["route"], classifier_arcs=pred["classifier_arcs"],
                            title=f"Student-test trip {tid}: HMM top-1 {sc['hmm_top1_accuracy']:.1%}, "
                                  f"classifier-only {sc['classifier_top1_accuracy']:.1%}")
        m.save(str(OUT / f"student_test_{tid}.html"))
        rows.append({"trip": tid, **sc, **pred["timings"]})
        print(tid, {k: round(v, 4) for k, v in sc.items()})
    pd.DataFrame(rows).to_csv(OUT / "maps_trip_scores.csv", index=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/PathSnap/scripts/07_refit_final_model.py
"""Step 7 (optional) - Re-fit the final model from the recorded best hyperparameters.

Use this on a new machine if models/final_model.joblib cannot be loaded
(e.g. a different scikit-learn version). It needs only the training data and
results/bayes_opt/best_params.json, and takes well under a minute.

Run:  python scripts/02_build_features.py   (if results/features/ is missing)
      python scripts/07_refit_final_model.py
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import joblib
from sklearn.ensemble import HistGradientBoostingClassifier

from pathsnap import config, training
from pathsnap.features import FEATURES


def main():
    best = json.load(open(config.RESULTS_DIR / "bayes_opt" / "best_params.json"))["best_params"]
    params = dict(best, early_stopping=False, random_state=config.RANDOM_SEED)
    tr = training.load_training_rows()
    m = HistGradientBoostingClassifier(**params).fit(tr[FEATURES].values, tr.y.values)
    config.MODELS_DIR.mkdir(exist_ok=True)
    joblib.dump(m, config.MODELS_DIR / "final_model.joblib", compress=3)
    json.dump({"model": "HistGradientBoostingClassifier", "params": params, "features": FEATURES,
               "candidate_radius_m": config.CANDIDATE_RADIUS_M, "max_candidates": config.MAX_CANDIDATES},
              open(config.MODELS_DIR / "model_config.json", "w"), indent=2)
    print("saved", config.MODELS_DIR / "final_model.joblib")


if __name__ == "__main__":
    main()

## Run the experiments
`run()` streams each script's output and stops the notebook if a step fails.

In [ ]:
def run(cmd):
    print(">>>", cmd, flush=True)
    env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"step failed: {cmd}"

### Step 1 - Exploratory data analysis

In [ ]:
run(f"python scripts/01_eda.py")

### Step 2 - Labels, candidates and features

In [ ]:
run(f"python scripts/02_build_features.py")

### Step 3 - Train and compare 7 models (5-fold GroupKFold CV + student test)

In [ ]:
run(f"python scripts/03_train_models.py")

### Step 3c - Comparison table and plots

In [ ]:
run(f"python scripts/03c_compare_models.py")

### Step 4 - Bayesian hyperparameter optimisation (Optuna TPE)

In [ ]:
run(f"python scripts/04_bayes_opt.py --trials {N_TRIALS}")

### Step 5 - HMM / Viterbi decoding experiments

In [ ]:
run(f"python scripts/05_hmm_decoding.py")

### Step 3b - Training histories (HGB per iteration, MLP per epoch)

In [ ]:
run(f"python scripts/03b_learning_curves.py")

### Step 6 - Folium maps of the final pipeline

In [ ]:
run(f"python scripts/06_make_maps.py")

## Key results

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
R = f"{REPO}/results"
print(open(f"{R}/eda/data_quality_notes.md").read())
cols = ["model", "family", "cv_macro_f1_mean", "cv_macro_f1_std", "train_macro_f1", "test_top1",
        "test_macro_f1", "test_log_loss", "test_route_continuity", "fit_seconds", "inference_ms_per_1k_obs"]
display(pd.read_csv(f"{R}/models/model_comparison.csv")[cols].round(4))
print(json.load(open(f"{R}/models/baseline_nearest_segment.json")))
print(json.load(open(f"{R}/bayes_opt/best_params.json")))
display(pd.read_csv(f"{R}/bayes_opt/tuned_vs_untuned.csv").round(4))
display(pd.read_csv(f"{R}/hmm/classifier_vs_hmm_student_test.csv").round(4))
display(pd.read_csv(f"{R}/hmm/student_test_per_trip_final_model.csv").round(4))
for img in ["models/cv_macro_f1_comparison.png", "models/student_test_comparison.png",
            "models/log_loss_comparison.png", "models/confusion_matrices.png",
            "models/training_histories.png", "bayes_opt/optimisation_history.png",
            "hmm/classifier_vs_hmm.png", "hmm/per_trip_student_test.png"]:
    print(img); display(Image(f"{R}/{img}"))

## Package the outputs (download `pathsnap_outputs.zip` from the Output tab)

In [ ]:
import sklearn
with open(f"{REPO}/models/training_environment.json", "w") as fh:
    json.dump({"python": platform.python_version(), "scikit-learn": sklearn.__version__,
               "numpy": numpy.__version__, "scipy": scipy.__version__, "platform": "Kaggle"}, fh, indent=2)

SKIP = ("results/features/train_candidates.parquet", "results/features/student_test_candidates.parquet",
        "results/features/train_observations.csv", "results/features/student_test_observations.csv")
zpath = "/kaggle/working/pathsnap_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in ["results", "models"]:
        for root, _, files in os.walk(os.path.join(REPO, top)):
            for f in files:
                full = os.path.join(root, f)
                rel = os.path.relpath(full, REPO).replace(os.sep, "/")
                if rel in SKIP or "/_labeldist_" in rel:
                    continue
                z.write(full, rel)
print(zpath, round(os.path.getsize(zpath) / 1e6, 1), "MB")